<a href="https://colab.research.google.com/github/lestermartin/starburst-dataframes-exploration/blob/main/Ibis/IbisWithStarburstGalaxy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Building Data Pipelines with Ibis and Starburst

This notebook represents the hands-on portion of the [Building Data Pipelines with Ibis and Starburst](https://github.com/starburstdata/devrel/tree/main/workshops/ibis-pipelines) webinar.

**Follow the [INSTRUCTIONS](https://github.com/starburstdata/devrel/blob/main/workshops/ibis-pipelines/INSTRUCTIONS.md) to get setup with Starburst Galaxy and a writeable catalog leveraging an S3 bucket.**


## Set YOUR env properties

The examples in the remainder of this notebook where build while using [Starburst Galaxy](https://www.starburst.io/starburst-galaxy/) (a super fast way to get going with Trino) using it's simple username/password authentication.  

**As stated above, follow the [INSTRUCTIONS](https://github.com/starburstdata/devrel/blob/main/workshops/ibis-pipelines/INSTRUCTIONS.md) to get setup with Starburst Galaxy and a writeable catalog leveraging an S3 bucket.**

To obtain the host name and your full user name, log into YOUR [Starburst Galaxy tenant](https://galaxy.starburst.io/login) and follow these steps.

1. From the *left-hand nav*, click on **Admin** > **Clusters**
2. From the **Clusters** list, scroll to the far right of the cluster you will be using and click on the *vertical elipsis* icon
3. Select **Partner connect** from the *contextual menu* that surfaces
4. Scroll down to the **Drivers & Clients** section and click on the Ibis tile
5. Use the **User** and **Host** properties present (plus your password) in the next cell

In [ ]:
import getpass

# grab credentials from the notebook user to be used when making a connection
my_host = input("Host name")
my_username = input("User name")
my_password = getpass.getpass("Password")

## Exploring Ibis

Python DataFrame API as described at the [Ibis project website](https://ibis-project.org/).

NOTE: Ibis can run against many different SQL engines, not just Trino.

### Env setup

In [ ]:
# install Ibis

%pip install trino
%pip install 'ibis-framework[trino]'

In [ ]:
# boiler-plate code for setup

import os
import ibis
from trino.auth import BasicAuthentication

ibis.options.interactive = True

user = my_username
trino_auth_obj = BasicAuthentication(my_username, my_password)
host = my_host
port = "443"
http_scheme = "https"
catalog = "tmp_cat"
schema = "tmp_lester_tester_12345"  #"tmp_first_last_postalcode" # don't forget to CHANGE this!

con = ibis.trino.connect(
    user=user, auth=trino_auth_obj, host=host, port=port, http_scheme=http_scheme, database=catalog, schema=schema
)

print('\n Make sure the phrase ** CONNECTION IS GOOD ** displays \n')
con.sql("select '** CONNECTION IS GOOD **' as conn_check")

### Interactive exploration

You could slide into your problem solving by take baby steps to see what each bit is doing for you. We can check the Galaxy query insights UI to see what the actual query that was sent over looked like.

Note: This code was originally published in  [ibis & trino (dataframe api part deux)](https://lestermartin.blog/2023/10/27/ibis-trino-dataframe-api-part-deux/).  

In [ ]:
# select a full table

custDF = con.table("customer", database=("tpch", "tiny"))
custDF[0:10]

In [ ]:
# use projection

projectedDF = custDF.select("name", "acctbal", "nationkey")
projectedDF[0:10]

In [ ]:
# apply a predicate

filteredDF = projectedDF.filter(projectedDF["acctbal"] > 50.0)
filteredDF[0:100]

In [ ]:
# select a second table

nationDF = con.table("nation", database=("tpch", "tiny")) \
            .drop("regionkey", "comment") \
            .rename(
                dict(
                    nation_name="name",
                    n_nationkey="nationkey"
                )
            )
nationDF[0:10]

In [ ]:
# join the tables

joinedDF = filteredDF.join(nationDF,
    filteredDF.nationkey == nationDF.n_nationkey)
joinedDF[0:10]

In [ ]:
# project the joined results

projectedJoinDF = joinedDF.drop("nationkey", "n_nationkey")
projectedJoinDF[0:10]

In [ ]:
# do some aggregations

groupedDF = projectedJoinDF.group_by("nation_name").aggregate(
    nbr_custs   = projectedJoinDF.count(),
    avg_acctbal = projectedJoinDF.acctbal.mean()
)
groupedDF[0:10]

In [ ]:
# add sorting

orderedDF = groupedDF.order_by([ibis.desc("avg_acctbal")])
orderedDF[0:10]

#### Leverage lazy execution

The baby steps above were great for figuring out your logic, but if you focus only getting the final answer, all of the DataFrame transformations will simply wait until you have a final I/O action triggered.

In [ ]:
# put it all together

nationDF = con.table("nation", database=("tpch", "tiny")) \
            .drop("regionkey", "comment") \
            .rename(
                dict(
                    nation_name="name",
                    n_nationkey="nationkey"
                )
            )

custDF = con.table("customer", database=("tpch", "tiny")) \
            .select("name", "acctbal", "nationkey") \
            .filter(projectedDF["acctbal"] > 50.0)

apiSQL = custDF.join(nationDF,
    custDF.nationkey == nationDF.n_nationkey) \
            .drop("nationkey", "n_nationkey") \
            .group_by("nation_name") \
            .aggregate(
                nbr_custs   = projectedJoinDF.count(),
                avg_acctbal = projectedJoinDF.acctbal.mean()
            ) \
            .order_by([ibis.desc("avg_acctbal")])

apiSQL[0:10]

#### SQL is available, too

Ibis strives to make their DataFrame API be the only (or at least primary) way you run your data operations, it sometimes it just might be better to write the SQL to get a different result and/or process to run.

NOTE: There could be a performance difference as Ibis is not doing anything with the SQL you write -- it just submits it like it is. Additionally, this approach hinders the optionality promise of Ibis due to SQL nuances among different SQL engines.

In [ ]:
# or... just run some SQL

sqlDF = con.sql("""
         SELECT n.name AS nation_name,
                COUNT() AS nbr_custs,
                AVG(c.acctbal) AS avg_acctbal
           FROM tpch.tiny.customer c
           JOIN tpch.tiny.nation n
             ON c.nationkey = n.nationkey
          WHERE c.acctbal > 50.0
          GROUP BY n.name
          ORDER BY AVG(c.acctbal) DESC
""")
sqlDF[0:10]

## Real-world example

Let's use the Burst Bank customer table as if it is our raw source already ingested. We can focus on doing some limited quality checks, and transformations, as well as a precalculated aggregates table.

### Explore the data

Notice how the optional `database=("<catalog>", "<schema>")` parameter allows you to select something different than was configured in the `con` object.

Seems a good idea to always being this specific; especially for code to be shared and/or productionalized.

Let's start with the `customer` table.

In [ ]:
bank_customer = con.table("customer", database=("sample", "burstbank"))
bank_customer[0:20]

#### Addresses

A quick glance suggests there are US and CA addresses which already (further) suggests that `state` is actually US-oriented.

Let's see if the country/state combos seem to make sense.

In [ ]:
state_groups = bank_customer.group_by(["country", "state"]).aggregate(
    nbr_custs = bank_customer.custkey.count(),
    avg_fico  = bank_customer.fico.mean()
).order_by("country", "state")

state_groups.to_pandas().head(2000)

67 rows sounds a little high as there are 50 states in the US and something less than 17 for Canadian provinces. Upon deeper look, this includes Canadian territories and US military designators for those serving overseas.

Generally, the data is PROBABLY good so we can leave it alone. Probably would want to verify that the upstream system is validating & standardizing addresses at capture time, but always a good data quality check. Could even build some error handling if it seemed necessary.

#### Phone numbers

The `phone` data looks to be all over the place. Let's install a phone number checking library and do a simple hard-coded test to make sure it works.

In [ ]:
%pip install phonenumbers

In [ ]:
import phonenumbers

def standardize_phone(raw: str, region: str = "US") -> str:
    parsed = phonenumbers.parse(raw, region)

    if not phonenumbers.is_valid_number(parsed):
        raise ValueError(f"Invalid phone number: {raw}")

    return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164)


# Examples
numbers = [
    "2025551234",
    "(202) 555-1234",
    "(202) 555-1234 ext. 103", # notice how extensions get lost
    "202-555-1234x103",
    "202.555.1234",
    "+1 202 555 1234",
    "1-202-555-1234",
]

for n in numbers:
    print(f"{n:25} -> {standardize_phone(n)}")

Pull out the phone number column from the dataframe and run the values against the generator to see what kind of shape this data is in.

In [ ]:
import pandas as pd
import phonenumbers

# quick wrapper around the phonenumbers library
def standardize_phone(raw: str, region: str = "US") -> str:
    try:
        parsed = phonenumbers.parse(str(raw), region)
        if not phonenumbers.is_valid_number(parsed):
            return None  # or return raw to keep original
        return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164)
    except phonenumbers.NumberParseException:
        return None  # unparseable values return None

# Trino does NOT support this kind of basic Python UDF (needs SQL)
phone_pdf = bank_customer.select("phone").to_pandas()

# With the pandas DF create a new column for the cleaned up values
phone_pdf["phone_std"] = phone_pdf["phone"].apply(standardize_phone)
phone_pdf.head(1500)

Most values are not getting standardizing, but it isn't as bad as it seems as this is a generated list of numbers and most actually are invalid for one reason or another.

We did find out from the earlier test of `phonenumbers` that the extensions get tossed away.

We determined:

1. We cannot reliably perform standardization with the bad data we have at this point.
1. We need to go upstream to enhance the data collection application to be more rigorous than just capturing a string.

IF the data was in good enough shape to standardize a high-percentage of the phone numbers, we could include that as an additional column on the silver table that could be created from this bronze dataset.



#### Dates

Below we see the dates all seem to be in good shape so we'll update the cast them into our silver table.

In [ ]:
check_dates = bank_customer.select("custkey", "dob", "registration_date").mutate(
    #key_casted = bank_customer.custkey.cast("date"), # uncomment to show all would blow up if bad
    dob_casted = bank_customer.dob.cast("date"),
    reg_casted = bank_customer.registration_date.cast("date")
)

print(check_dates)

#### Booleans

We find out below the Y/N values are consistent in the direct deposit flag field so we'll convert them to real booleans for our silver table.

In [ ]:
check_direct_deposit = bank_customer.select("custkey", "paycheck_dd").mutate(
    direct_deposit = ibis.case()
        .when(bank_customer.paycheck_dd.upper().left(1) == "Y", True)
        .when(bank_customer.paycheck_dd.upper().left(1) == "N", False)
        .else_(None)       # anything unexpected becomes NULL
        .end()
        .cast("boolean")
).group_by("direct_deposit").aggregate(
    nbr_custs = bank_customer.paycheck_dd.count()
)

print(check_direct_deposit)

### Create silver dataset

We'll just convert the string dates to actual Date columns and double-check the Y/N values for direct deposit before casting this to a boolean. This is to show we did SOME cleanup/transformation activities, and the write it as an Iceberg table. This is just a simple example of bronze to silver pipelining.

In [ ]:
# perform all transformations

silver_t = con.table("customer", database=("sample", "burstbank")).mutate(
    dob = bank_customer.dob.cast("date"),
    registration_date = bank_customer.registration_date.cast("date"),
    paycheck_dd = ibis.case()
        .when(bank_customer.paycheck_dd.upper().left(1) == "Y", True)
        .when(bank_customer.paycheck_dd.upper().left(1) == "N", False)
        .else_(None)       # anything unexpected becomes NULL
        .end()
        .cast("boolean")
)
silver_t[0:10]

In [ ]:
# Create the table and write the first batch into it

# leaving off the database=("cat", "sch") parameter to use what con was configured with
con.create_table("customer_silver", silver_t, overwrite=True)
print("Table created!")

con.table("customer_silver").head(50)

In [ ]:
# Move to using the insert function instead of create_table

# get the BEFORE insert count
before_t = con.table("customer_silver")
row_count = before_t.count().execute()
print(f"Before count: {row_count}")


# add the same 1000 records again
con.insert("customer_silver", silver_t)


# get the AFTER insert count
after_t = con.table("customer_silver")
row_count = after_t.count().execute()
print(f"After count: {row_count}")

### Create gold dataset

Let's define a view that groups customers by their medallion status and the state/province they lives in and calculates the following for the group:

- Number of auto loans
- Average auto payment


In [ ]:
# grab needed fields from customer
all_cust = con.table("customer", database=("sample", "burstbank")) \
    .select("custkey", "state")

# grab needed fields from customer_profile
cust_p = con.table("customer_profile", database=("sample", "burstbank")) \
    .select("custkey", "customer_segment")

# css -> Customer State (and) Segment
css = all_cust.join(cust_p, all_cust.custkey == cust_p.custkey)

# grab needed fields from account
all_acct = con.table("account", database=("sample", "burstbank")) \
    .select("custkey", "auto_loan_id", "auto_loan_status", "auto_loan_balance")

# only keep those with ACTIVE auto loans
active_auto = all_acct.filter(
    [all_acct.auto_loan_status == "open",
     all_acct.auto_loan_balance > 0])

# grab needed fields from loan_payment
auto_pymt = con.table("auto_loan_payment", database=("sample", "burstbank")) \
    .select("auto_loan_id", "payment_amount")

# put it all together
the_aggs = active_auto.join(css, active_auto.custkey == css.custkey) \
    .join(auto_pymt, active_auto.auto_loan_id == auto_pymt.auto_loan_id) \
    .group_by(css.state, css.customer_segment) \
    .aggregate(
        nbr_auto_loans = active_auto.auto_loan_id.count(),
        avg_payment = auto_pymt.payment_amount.mean()
    )

# few touch-ups to be DONE
the_view = the_aggs.mutate(
    avg_payment = the_aggs.avg_payment.round()
).order_by("customer_segment", "state")


print(the_view)

That might have looked like a lot for those not used to Dataframe programming, but it wasn't all that short on SQL side either.

**In many ways... it's a matter of style & preference for the data engineer...**

In [ ]:
# or... just run some SQL

sql_version = con.sql("""

WITH

active_auto_loans AS (
SELECT custkey, auto_loan_id, auto_loan_balance
  FROM sample.burstbank.account
 WHERE auto_loan_status = 'open'
   AND auto_loan_balance > 0
),

cust_state_and_segment AS (
SELECT c.custkey, c.state, cp.customer_segment
  FROM sample.burstbank.customer c
  JOIN sample.burstbank.customer_profile cp
    ON c.custkey = cp.custkey
),

cust_active_auto AS (
SELECT *
  FROM active_auto_loans aal
  JOIN cust_state_and_segment csas
    ON aal.custkey = csas.custkey
)


SELECT caa.state, caa.customer_segment,
       count() AS nbr_auto_loans,
       round(avg(alp.payment_amount)) AS avg_payment
  FROM cust_active_auto caa
  JOIN sample.burstbank.auto_loan_payment alp
    ON caa.auto_loan_id = alp.auto_loan_id
 GROUP BY caa.state, caa.customer_segment
 ORDER BY caa.customer_segment, caa.state

""")
sql_version[0:10]

Finally, let's save it as a view instead of a fully created/populated table.

In [ ]:
# create the darn view
con.create_view("auto_loan_aggs_view", the_view  #, overwrite=True) # if you want to recreate it

# verify it is there
con.table("auto_loan_aggs_view").head(10)